# 03 · Agreement, adjudication → *your* gold set

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/03_annotate.ipynb)

The part no model can do for you, and the part the Q&A will ask about.

```
  01_build_pool_<track>  →  02_sample  →▶ 03_annotate  →  04_prompt  →  05_report
```

| | |
|---|---|
| **Reads** | `data/gold/<track>_<group>_sample.json` and the sheet (both from 02) |
| **Writes** | `data/gold/<track>_<group>_gold.json` |

---

**Come here when both coders have finished.** Notebook 02 drew the sample and made the sheet; this one turns two people's labels into one gold set.

The published labels are somebody else's judgment. You re-annotated the sample blind; now you find out how far apart the two of you were, and argue out the rows you disagreed on.

What comes out is *your* gold set — and the disagreements tell you which label boundaries are genuinely fuzzy. That is what lets you say, later, whether a model's miss is the **model's** fault or the **scheme's**. Nothing else in the project can tell you that, and notebook 05 asks you for it directly.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, MEMBERS,
                    LABELS_ORDER, ROOT, OUT_DIR, POOL_PATH, DEMO_POOL_PATH,
                    SAMPLE_PATH, GOLD_PATH, PRED_PATH, ROUNDS_PATH,
                    PROMPT_FILE, SHEET_PATH, TRIAGE_PATH, describe)

# The Google Sheets round trip is plumbing, so it is imported. The judgment it
# exists to support is not in any of these files.
from pipeline import load_gold, label_set, save_json
from annotate import (remembered_sheet, load_annotation_sheet, to_canonical,
                      annotator_agreement, disagreements,
                      compare_to_published)

describe()                  # what this notebook is working on


> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

## First — your sample, back from the file

Notebook 02 saved it, and this is the moment that was for. Days have passed, the runtime that drew the sample is long gone, and the person running this cell may not be the person who ran 02.

It matters that this is a **load and not a redraw**: the sheet your coders filled in was built from these exact forty items, and adjudication puts their labels back onto them one by one. `to_canonical` also uses this list to restore what the sheet does not carry — on `cars50` and `raamove`, the passage each sentence came from.

In [ ]:
sampled = load_gold(SAMPLE_PATH)
LABELS = label_set(sampled)

print(len(sampled), "items ·", LABELS)

# These still carry the PUBLISHED label. Ignore it for now — you compare
# against it in step 4, once your own labels are settled.


In [ ]:
# The sheet notebook 02 created, read back from the file it wrote. This is why it
# wrote one: whoever runs this notebook need not be the person who ran
# and need not still have that cell's output on screen.
SHEET_ID = remembered_sheet(SHEET_PATH)

# Working on a sheet someone made before this file existed? Paste its URL (or
# just the long id from it) here instead:
# SHEET_ID = ""

ROUND = "round1"             # each re-annotation round gets its own tab

print("sheet:", SHEET_ID or "-- none saved yet: run notebook 02 --")


## Step 1 — Measure agreement

Two numbers and a matrix: raw percent agreement, Cohen's κ (agreement corrected for what you would get by guessing), and an annotator-vs-annotator confusion matrix whose off-diagonal cells show *which* label pairs the two of you confuse.

**Write these down now** — they are report section 1, and they do not survive a runtime reset. A κ around .8 is strong; around .4 means the scheme, not the annotators, is doing something wrong. Either is a reportable finding. A low κ you can explain beats a high one you cannot.

In [ ]:
# ══ STEP 1 · Measure agreement ════════════════════════════════════════════
# Goal      : how often did the two of you agree, and on which labels did you not?
# Available : load_annotation_sheet(SHEET_ID, ROUND)  ->  rows
#             annotator_agreement(rows)  ·  disagreements(rows)
#             SHEET_ID · ROUND   (both set in the cell just above)
# Source    : scripts/annotate.py · load_annotation_sheet, annotator_agreement
# Pointer   : Day 2 S5 steps D–E — identical calls.
# Produce   : rows · disagreed      ← later cells use these names
# Note      : run this once BOTH CoderA and CoderB columns are filled in.
#             Half-finished rows are dropped from the comparison.
# Keep      : `disagreed` matters again in notebook 05 — the rows you
#             argued about are the ones to check your model's errors
#             against. Save the list, or write the ids down.

# ✏️ your code here — fill in each ____

rows = load_annotation_sheet(SHEET_ID, ____)     # ROUND

annotator_agreement(rows)                        # prints % agreement and κ
disagreed = disagreements(rows)
disagreed


## Step 2 — Adjudicate

Go back to the sheet and fill in `Final` for **every** row:

- Where you agreed, `Final` is that label.
- Where you did not, talk it out and decide. If you cannot agree, the scheme is underspecified — write down *why* in `Note` and pick one. That note is worth more to your report than the label is.

Then re-read the sheet and canonicalise it. `to_canonical` reports blanks and invalid labels rather than silently dropping them; fix them in the sheet and re-run until it says **0 blank, 0 invalid**.

In [ ]:
# ══ STEP 2 · Adjudicate, then canonicalise ════════════════════════════════
# Goal      : agree a Final label for every row, then turn the sheet into gold.
# Available : load_annotation_sheet(SHEET_ID, ROUND)  ->  rows   (re-read after editing)
#             to_canonical(rows, LABELS, source=sampled)  ->  gold
# Source    : scripts/annotate.py · to_canonical
# Pointer   : Day 2 S5 step F — identical calls.
# Produce   : gold      ← later cells use these names
# Careful   : re-read the sheet first. `rows` from step 1 is a snapshot
#             from before you filled in Final.
# Note      : keep going until it prints 0 blank and 0 invalid. A blank
#             row is an item silently missing from your study.
# Note      : pass source=sampled. Gold is rebuilt from the SHEET, which
#             holds only the id, the text and your label — anything else
#             the item carried (on cars50/raamove, its passage) is put
#             back from `sampled` by id. Harmless on the other tracks.

# ✏️ your code here — fill in each ____

# Re-read: `rows` from step 3 was fetched before you filled in Final.
rows = load_annotation_sheet(SHEET_ID, ROUND)

gold = to_canonical(rows, LABELS, source=____)      # sampled


## Step 3 — Where do you differ from the published labels?

Now — and only now, with your own labels settled — look at what the corpus said. `compare_to_published` matches by text and shows you every row where your group landed somewhere else.

**Disagreement here is not an error.** You annotated forty items carefully against a scheme you had thought about; the original annotators worked at scale under different guidelines. Where you differ, one of three things is true, and saying which is exactly the analytical work this project is for:

1. **Your scheme drifted** from theirs — you read a category boundary differently. Say where.
2. **The item is genuinely ambiguous** — it would split any pair of annotators.
3. **One of you is wrong.** It happens, in both directions.

This table is report section 1, and it is the one that most often produces a sentence worth saying out loud in the Q&A.

In [ ]:
# ══ STEP 3 · Compare against the published labels ═════════════════════════
# Goal      : see where your gold set and the corpus disagree, and work out why.
# Available : compare_to_published(gold, sampled)  ->  a table of the rows that differ
# Source    : scripts/annotate.py · compare_to_published
# Pointer   : Day 2 S5 step F — the same call.
# Produce   : differences      ← later cells use these names
# Careful   : compare against `sampled`, not `pool`. Sampling renumbers
#             the ids, so `pool` would line your item 7 up against a
#             different sentence entirely.
# Note      : pick two or three and write down which of the three cases
#             above they are. Do it now, while you remember the argument.

# ✏️ your code here — fill in each ____

differences = compare_to_published(gold, ____)      # sampled — not pool
differences


## Save it — this is the handoff

This file is the single most valuable thing your group makes all week — hours of judgment, and the only thing in the project that could not have been produced by a script. Every number in notebooks 04 and 05 is measured against it, and it goes in your submission bundle.

**Next:** open `04_prompt.ipynb`. It starts by loading `data/gold/<track>_<group>_gold.json`.

In [ ]:
save_json(gold, GOLD_PATH, what="gold items")

# It is git-ignored — it is your work, not part of the template. If you cloned
# into Google Drive it is already saved across sessions; if not, download it.


---

## 🛑 The `PLAN.md` gate

Notebook 04 starts calling the model. **Do not open it until your `PLAN.md` has been read and signed off.** It takes two minutes and it is not busywork: a mismatched label set or an unstated sampling seed costs an hour to unpick *after* you have burned quota on it.

Check, out loud, that these three agree: the label set in `PLAN.md`, the labels `label_set` actually returned above, and the labels your prompt file names. And that `PLAN.md` records **which sampling strategy you chose, and why**.